In [1]:
# step5_manifest_check_with_logs.py

import os
import base64
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from dotenv import load_dotenv
from time import sleep, time as now

# === Load GitHub Tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 6)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")

token_index = 0
def get_headers():
    return {
        "Authorization": f"token {tokens[token_index]}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "manifest-checker"
    }

def rotate_token():
    global token_index
    token_index = (token_index + 1) % len(tokens)
    print(f"🔁 Rotated to token #{token_index + 1}")

# === Make GitHub API Request with Retry and Rotation ===
def make_request(url):
    tries_flag = 0
    while True:
        res = requests.get(url, headers=get_headers())
        if res.status_code == 200:
            return res
        elif res.status_code == 403 and "rate limit" in res.json().get("message", "").lower():
            rotate_token()
            tries_flag += 1
            if tries_flag > len(tokens):
                reset_timestamp = int(res.headers.get("X-RateLimit-Reset", now() + 60))
                wait_time = reset_timestamp - int(now())
                print(f"⏳ Rate limit hit. Sleeping for {wait_time} seconds...")
                sleep(wait_time + 1)
                tries_flag = 0
            else:
                sleep(1)
        elif res.status_code in [404, 422]:
            return None
        else:
            print(f"⚠️ Request failed: {res.status_code}. Retrying...")
            sleep(1)

# === Check if manifest and activity exist ===
def check_manifest_and_activity(full_name):
    owner_repo = full_name.strip()
    search_url = f"https://api.github.com/search/code?q=filename:AndroidManifest.xml+repo:{owner_repo}"
    search_res = make_request(search_url)
    if not search_res:
        return "no", "no", "no"

    data = search_res.json()
    if data.get("total_count", 0) == 0:
        return "no", "no", "no"

    found_activity = False
    found_standard = False
    found_any = False

    for item in data.get("items", []):
        path = item.get("path", "").lower()
        found_any = True

        is_standard = path in ["app/src/main/androidmanifest.xml", "src/main/androidmanifest.xml"]
        if is_standard:
            found_standard = True
            print(f"📁 Found standard manifest path: {path}")
        else:
            print(f"📄 Found non-standard manifest path: {path}")

        file_url = item.get("url")
        file_res = make_request(file_url)
        if not file_res:
            continue

        content = file_res.json().get("content")
        if not content:
            continue

        try:
            xml_text = base64.b64decode(content).decode("utf-8", errors="ignore")
            root = ET.fromstring(xml_text)
            activity_nodes = root.findall(".//activity")
            if activity_nodes:
                print(f"✅ Found <activity> tag(s) in: {path}")
                found_activity = True
                break
            else:
                print(f"❌ No <activity> tag found in: {path}")
        except ET.ParseError:
            print(f"⚠️ XML parsing failed for: {path}")
            continue

    return (
        "yes" if found_any else "no",
        "yes" if found_activity else "no",
        "yes" if found_standard else "no"
    )

# === Paths ===
input_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step4_keyword_check_output.csv"
output_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step5_manifest_check_output.csv"

# === Load CSV and filter ===
df = pd.read_csv(input_path)
df["has_manifest"] = "N/A"
df["has_activity"] = "N/A"
df["standard_manifest"] = "N/A"

filtered_df = df[df["keyword_check"] == "pass"].copy()
filtered_indices = filtered_df.index.tolist()

# === Process filtered repos ===
for idx, original_index in enumerate(filtered_indices):
    full_name = df.loc[original_index, "full_name"]
    print(f"\n🔍 [{idx+1}/{len(filtered_indices)}] Checking: {full_name}")

    has_manifest, has_activity, standard_manifest = check_manifest_and_activity(full_name)

    df.at[original_index, "has_manifest"] = has_manifest
    df.at[original_index, "has_activity"] = has_activity
    df.at[original_index, "standard_manifest"] = standard_manifest

    if (idx + 1) % 10 == 0:
        interim_path = output_path.replace(".csv", f"_interim_{idx+1}.csv")
        df.to_csv(interim_path, index=False)
        print(f"💾 Interim results saved at: {interim_path}")

# === Save final results ===
df.to_csv(output_path, index=False)
print(f"\n✅ Step 5 complete. Results saved to: {output_path}")



🔍 [1/23470] Checking: ligi/gobandroid
📄 Found non-standard manifest path: android/src/main/androidmanifest.xml
✅ Found <activity> tag(s) in: android/src/main/androidmanifest.xml

🔍 [2/23470] Checking: quran/quran_android
📁 Found standard manifest path: app/src/main/androidmanifest.xml
✅ Found <activity> tag(s) in: app/src/main/androidmanifest.xml

🔍 [3/23470] Checking: facebook/facebook-android-sdk
📄 Found non-standard manifest path: facebook/src/main/androidmanifest.xml
❌ No <activity> tag found in: facebook/src/main/androidmanifest.xml
📄 Found non-standard manifest path: samples/kotlinsampleapp/androidmanifest.xml
✅ Found <activity> tag(s) in: samples/kotlinsampleapp/androidmanifest.xml

🔍 [4/23470] Checking: thunderbird/thunderbird-android
📄 Found non-standard manifest path: app-k9mail/src/main/androidmanifest.xml
✅ Found <activity> tag(s) in: app-k9mail/src/main/androidmanifest.xml

🔍 [5/23470] Checking: JetBrains/ideavim

🔍 [6/23470] Checking: wuan/bo-android
📁 Found standard man